# EuroCropsML: EDA

EuroCropsML is the crop-type classification benchmark in this repo. It covers 706,683 agricultural fields across Estonia, Latvia, and Portugal, each labelled with an HCAT crop code (coarsened to ~36 types at 6 digits). The task is transnational transfer: train on Latvia + Portugal, test on Estonia.

The dataset is **S2-only** — each field has an irregular multi-date Sentinel-2 time series with no accompanying Sentinel-1 SAR or climate data (those arrays are zero-filled by the loader). A sample is a single field (called a *parcel* in the dataset), stored as a polygon rather than a single pixel: each acquisition date stores the median reflectance across all S2 pixels inside that field boundary. Fields range from <1 ha to hundreds of hectares, so each sample represents anywhere from a handful to tens of thousands of 10 m S2 pixels.


## Setup

In [ ]:
from pathlib import Path
import glob, json
import numpy as np
import matplotlib.pyplot as plt

# find the repo root by walking up to the folder that has data/
REPO = Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent

EC = REPO / 'data' / 'input' / 'benchmarks' / 'eurocropsml' / 'preprocess'
files = sorted(glob.glob(str(EC / '*.npz')))
print(len(files), 'EuroCropsML parcels')
# country is the filename prefix; HCAT crop code is the last underscore field
country_of = {'EE': 'Estonia', 'LV': 'Latvia', 'PT': 'Portugal'}

The full native time series is passed through to the encoders (no resampling). EuroCropsML has no native Sentinel-1 SAR or climate data, so those modalities are filled with zeros.


In [ ]:
path = files[0]
d = np.load(path)
data, dates, center = d['data'].astype(float), d['dates'], d['center']
code = Path(path).stem.split('_')[-1]      # HCAT crop code
country = country_of.get(Path(path).stem[:2], Path(path).stem[:2])
print('file:', Path(path).name)
print('shape:', data.shape, '| timesteps:', len(dates), '| crop HCAT:', code, '| country:', country)
print('first dates:', list(dates[:4]))

# show the raw data as a table: timesteps as rows, S2 bands as columns
# Native Sentinel-2 band order: B1..B12, no SCL column (verified 2026-06-14)
S2_BANDS = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B10','B11','B12']
t = pd.DataFrame(data, columns=S2_BANDS)
t.insert(0, 'date', dates)
t.round(4).head(8)   # first 8 acquisitions (space)


### What each column means

### What each column means

The raw npz files store the 13 native Sentinel-2 bands in order (B1..B12, **no SCL column**).
The loader's `EC_S2_IDXS` picks B2..B8A (idx 1-8), B11 (idx 11), B12 (idx 12), then appends NDVI.
B1, B9, and B10 are dropped — no encoder uses them.

| idx | Band | Source | Used by code? |
|:---:|------|--------|:-------------:|
| 0 | B1 | Sentinel-2 MSI (Coastal aerosol, 60m) | ❌ Skipped |
| 1 | B2 | Sentinel-2 MSI (Blue, 10m) | ✅ S2 |
| 2 | B3 | Sentinel-2 MSI (Green, 10m) | ✅ S2 |
| 3 | B4 | Sentinel-2 MSI (Red, 10m) | ✅ S2 |
| 4 | B5 | Sentinel-2 MSI (Red Edge 1, 20m) | ✅ S2 |
| 5 | B6 | Sentinel-2 MSI (Red Edge 2, 20m) | ✅ S2 |
| 6 | B7 | Sentinel-2 MSI (Red Edge 3, 20m) | ✅ S2 |
| 7 | B8 | Sentinel-2 MSI (NIR, 10m) | ✅ S2 |
| 8 | B8A | Sentinel-2 MSI (Narrow NIR, 20m) | ✅ S2 |
| 9 | B9 | Sentinel-2 MSI (Water vapour, 60m) | ❌ Skipped |
| 10 | B10 | Sentinel-2 MSI (Cirrus, 60m) | ❌ Skipped (near-zero over land) |
| 11 | B11 | Sentinel-2 MSI (SWIR 1, 20m) | ✅ S2 |
| 12 | B12 | Sentinel-2 MSI (SWIR 2, 20m) | ✅ S2 |

The code's `EC_S2_IDXS` picks indices `[1..8, 11, 12]` (B2..B8A, B11, B12) then appends a computed NDVI = `(B8-B4)/(B8+B4)`, yielding **11 S2 bands**. S1 and climate are zero-filled — EuroCropsML is **S2-only**.

## What this one parcel looks like over its season

Bands are at raw reflectance scale (B4 = red, B8 = NIR); we add the NDVI it implies.

In [ ]:
# band order in the npz: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B10,B11,B12 (native S2)
doy = dates.astype('datetime64[D]')
b4, b8 = data[:, 3], data[:, 7]
ndvi = np.where((b8 + b4) > 0, (b8 - b4) / (b8 + b4), 0)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for c, n in zip([1, 2, 3, 7, 11, 12], ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']):
    ax[0].plot(doy, data[:, c], marker='.', label=n)
ax[0].set_title('Sentinel-2 reflectance'); ax[0].set_ylabel('reflectance'); ax[0].legend(ncol=6, fontsize=8)
ax[1].plot(doy, ndvi, marker='.', color='green')
ax[1].set_title('NDVI'); ax[1].set_ylabel('NDVI'); ax[1].axhline(0, color='grey', lw=0.6)
fig.suptitle(f'one {country} parcel  (HCAT {code})')
plt.tight_layout(); plt.show()

## Country mix and irregular sequence lengths

We sample a few thousand filenames (cheap) rather than opening every parcel.

In [ ]:
import collections
rng = np.random.default_rng(0)
sample = [files[i] for i in rng.permutation(len(files))[:6000]]
countries = [country_of.get(Path(p).stem[:2], '?') for p in sample]
lengths = []
for p in sample[:1500]:
    lengths.append(len(np.load(p)['dates']))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cc = collections.Counter(countries)
ax[0].bar(list(cc), list(cc.values()), color=['#2563eb', '#16a34a', '#f97316'])
ax[0].set_title('parcels by country (sample of 6000)')
ax[1].hist(lengths, bins=30, color='#7c3aed')
ax[1].set_title('timesteps per parcel'); ax[1].set_xlabel('# acquisitions')
plt.tight_layout(); plt.show()

## Crop-type granularity (why we coarsen HCAT)

The full 10-digit HCAT code is hierarchical: the **first 6 digits** encode the coarse crop family (e.g., cereal, legume, grass), and the last 4 encode fine variety/detail. The `crop-class` task truncates to the first 6 digits (~36 crop types) to avoid hundreds of long-tail variety classes with only a few examples each. Here is the long tail.


In [ ]:
codes = [Path(p).stem.split('_')[-1] for p in sample]
full = collections.Counter(codes)
coarse = collections.Counter(c[:6] for c in codes)
print('full HCAT classes:', len(full), '| coarsened (6-digit):', len(coarse))
top = coarse.most_common(15)
plt.figure(figsize=(11, 4))
plt.bar([k for k, _ in top], [v for _, v in top], color='#0ea5e9')
plt.title('top coarsened crop types (6-digit HCAT)'); plt.xticks(rotation=60); plt.ylabel('parcels')
plt.tight_layout(); plt.show()

## Takeaways

One sample is an irregular S2-only parcel time series (no native S1 / climate). Label = HCAT crop code, coarsened to ~36 types for the task. Country is the holdout group: train Latvia + Portugal, test Estonia.